# RFP vs Proposal Coverage Mapper: Day 4 (retrieval + LLM judge)

**Before running:**
1. **Add Input**: attach ONLY the dataset `rfp-coverage-mapper-final`
2. **Settings > Internet**: On (needed to download the embedding model) · **Accelerator**: None
3. **Add-ons > Secrets**: tick `GEMINI_API_KEY`

Then click **Run All** (about 5–10 minutes; Step 8 makes ~21 Gemini calls, spaced 6 seconds apart).
Every step ends with **✅ STEP X DONE** or **❌ STEP X FAILED: reason**. A failed step stops the run on purpose.

## Step 1: Find the project and copy it

In [ ]:
# ===== STEP 1: find the project in the dataset and copy it to the working folder =====
import os, sys, glob, shutil

class StepFailed(Exception):
    pass

def fail(step, reason):
    print(f'❌ STEP {step} FAILED: {reason}')
    raise StepFailed(reason)   # stops Run All on purpose

PROJECT = '/kaggle/working/rfp-coverage-mapper'
candidates = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/check_setup.py', recursive=True)]
complete = [c for c in candidates if os.path.isfile(os.path.join(c, 'test_data/inputs/proposal_qamar.pptx'))]
print('Folders with check_setup.py:', candidates or 'none')
if not complete:
    fail(1, 'no project folder with test_data/ found. Is the dataset "rfp-coverage-mapper-final" attached (Add Input)?')
SOURCE = sorted(complete, key=lambda c: ('final' not in c, len(c)))[0]
print('Project found at:', SOURCE)
shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(SOURCE, PROJECT)
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
for folder in ('evaluation', 'cache', 'output'):
    os.makedirs(folder, exist_ok=True)
print('Working in:', os.getcwd())
print('✅ STEP 1 DONE: project copied to', PROJECT)

## Step 2: Save the code files
Each cell saves one file (`Writing ...` / `Overwriting ...`). New today: `retrieve.py`, `judge.py`, `check_coverage.py`; `llm.py` gains call pacing and sticky backup. The last file is the official Day 3 extraction (cached Gemini answer), so the 21 requirements stay identical.

In [ ]:
%%writefile requirements.txt
google-genai        # official Gemini SDK
python-dotenv       # loads the API key from .env
python-docx         # read/write Word files
python-pptx         # read/write PowerPoint files
pdfplumber          # read PDF files (Day 2)
reportlab           # write PDF files (synthetic RFP)
pandas              # tables (Day 5)
openpyxl            # Excel output (Day 5)
sentence-transformers  # local embedding model for slide retrieval (Day 4)


In [ ]:
%%writefile src/llm.py
"""
llm.py - the ONE place in the project that talks to Gemini.

Every other file calls ask_json(). Keeping all AI calls here means:
  - the API key is loaded in one place (never written in code)
  - busy/rate-limit errors are handled once, with an automatic backup model
    (if the main model gives up, the backup is used for the rest of the run)
  - every answer is cached to disk, so reruns are free and repeatable
  - switching model/provider later means changing only this file
"""
import hashlib
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # reads a .env file if one exists (local use); harmless if not

MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
# Backup model(s), used only if the main one keeps failing (busy / quota)
FALLBACK_MODELS = [m.strip() for m in os.getenv("GEMINI_FALLBACK_MODELS", "gemini-3.5-flash-lite").split(",") if m.strip()]
CACHE_DIR = Path(os.getenv("LLM_CACHE_DIR", "cache"))
USE_CACHE = os.getenv("LLM_CACHE", "on").lower() != "off"
ATTEMPTS_PER_MODEL = 3
# Free-tier friendly pacing: at least this many seconds between real API calls (10 calls per minute max)
MIN_SECONDS_BETWEEN_CALLS = float(os.getenv("LLM_MIN_INTERVAL", "6"))

_client = None
_key_source = None
_main_model_busy = False     # becomes True once the main model has failed 3 times in this run
last_model_used = None       # which model produced the most recent answer (or "cache: <model>")
_last_call_time = 0.0


def _find_api_key():
    """Return (key, where_it_came_from). Checks .env/environment, then Kaggle Secrets."""
    key = os.getenv("GEMINI_API_KEY")
    if key and key != "paste-your-key-here":
        return key, ".env file / environment variable"
    try:
        from kaggle_secrets import UserSecretsClient  # only exists on Kaggle
        key = UserSecretsClient().get_secret("GEMINI_API_KEY")
        if key:
            return key, "Kaggle Secrets"
    except Exception:
        pass
    return None, None


def get_client():
    """Create the Gemini client once and reuse it."""
    global _client, _key_source
    if _client is None:
        key, _key_source = _find_api_key()
        if not key:
            raise RuntimeError(
                "No GEMINI_API_KEY found. Locally: add it to .env. "
                "On Kaggle: Add-ons > Secrets, add GEMINI_API_KEY and tick it for this notebook."
            )
        _client = genai.Client(api_key=key)
    return _client


def key_source():
    """Where the API key was loaded from (call after get_client())."""
    return _key_source


def _cache_file(prompt, system, temperature, tag):
    """
    A cache 'fingerprint': the same main model + prompt + settings always gives
    the same file name. If anything changes (even one word), it's a new entry.
    """
    raw = json.dumps({"model": MODEL, "system": system, "prompt": prompt,
                      "temperature": temperature, "tag": tag}, sort_keys=True)
    return CACHE_DIR / (hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16] + ".json")


def _is_temporary(msg):
    """Errors worth waiting for: rate limits (429) and busy servers (503/504)."""
    signs = ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "overloaded", "504", "DEADLINE_EXCEEDED")
    return any(s in msg for s in signs)


def ask_json(prompt, system=None, temperature=0.2, cache_tag="", use_cache=True):
    """
    Send a prompt to Gemini and get back a Python dict/list.

    1. If we've asked this exact question before, return the saved answer (no API call).
    2. Otherwise call the main model in JSON mode, so the reply is machine-readable.
    3. Temporary error (busy / rate limit)? Wait and retry, doubling the wait
       each time ("exponential backoff"), and print the REAL reason.
    4. Still failing after 3 tries? Switch to the backup model and repeat.
    5. Save the answer to the cache for next time (noting which model answered).

    cache_tag lets us deliberately ask the same prompt again for a fresh answer
    (e.g. "deck-attempt-2" when regenerating a deck that broke the rules).
    """
    global _main_model_busy, last_model_used, _last_call_time
    caching = USE_CACHE and use_cache
    path = _cache_file(prompt, system, temperature, cache_tag)
    if caching and path.exists():
        saved = json.loads(path.read_text(encoding="utf-8"))
        last_model_used = f"cache: {saved.get('model_used', 'unknown')}"
        print(f"  [llm] cache hit ({path.name}) - no API call used")
        return saved["response"]

    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json",
    )
    models = [MODEL] + FALLBACK_MODELS
    if _main_model_busy and FALLBACK_MODELS:
        models = FALLBACK_MODELS   # main model already gave up earlier in this run: go straight to the backup
    for model in models:
        wait = 10
        for attempt in range(1, ATTEMPTS_PER_MODEL + 1):
            pause = MIN_SECONDS_BETWEEN_CALLS - (time.time() - _last_call_time)
            if pause > 0:
                time.sleep(pause)           # stay under the free tier's requests-per-minute limit
            _last_call_time = time.time()
            try:
                response = get_client().models.generate_content(model=model, contents=prompt, config=config)
                if not response.text:
                    raise ValueError("empty reply")
                result = json.loads(response.text)
                last_model_used = model
                if model != MODEL:
                    print(f"  [llm] answered by backup model {model}")
                if caching:
                    CACHE_DIR.mkdir(parents=True, exist_ok=True)
                    path.write_text(json.dumps({"model_used": model, "tag": cache_tag,
                                                "prompt_preview": prompt[:300],
                                                "response": result}, indent=2), encoding="utf-8")
                return result
            except (json.JSONDecodeError, ValueError):
                print(f"  [llm] {model}: reply was empty or not valid JSON (try {attempt}), retrying...")
            except Exception as e:
                msg = str(e)
                if not _is_temporary(msg):
                    raise  # a real error (bad key, bad model name...) - stop and show it
                print(f"  [llm] {model}: busy / rate limited (try {attempt}). Reason: {msg[:150]}")
                if attempt < ATTEMPTS_PER_MODEL:
                    print(f"        waiting {wait}s...")
                    time.sleep(wait)
                    wait *= 2
        print(f"  [llm] giving up on {model}, trying the next model...")
        if model == MODEL:
            _main_model_busy = True
            print("  [llm] the backup model will be used directly for the rest of this run")
    raise RuntimeError("All models failed. Wait a few minutes (free-tier limits reset) and run the cell again.")


In [ ]:
%%writefile src/parsers.py
"""
parsers.py - STAGE 1 of the tool: turn the input files into clean, structured text.

    parse_rfp(path)  -> the RFP (PDF or Word) as a list of "blocks"
    parse_deck(path) -> the proposal deck (PPTX) as a list of slides

A "block" is one heading, paragraph, bullet or table, tagged with the RFP
section it belongs to. Keeping this structure (instead of one big string)
means later stages can say WHERE in the RFP a requirement came from.

No AI is used here: parsing is deterministic, fast, free and repeatable.
"""
import re
from pathlib import Path

import pdfplumber
from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE

# ---------------------------------------------------------------------------
# Safety: the tool must never read the answer key (prevents data leakage)
# ---------------------------------------------------------------------------
BLOCKED_FOLDERS = {"ground_truth"}


def check_allowed(path):
    """Refuse files inside ground_truth/ and files that don't exist."""
    p = Path(path)
    if any(part in BLOCKED_FOLDERS for part in p.parts):
        raise PermissionError(f"Refusing to read '{p}': the tool must never read the answer key (ground_truth).")
    if not p.is_file():
        raise FileNotFoundError(f"File not found: '{p}'")
    return p


# ---------------------------------------------------------------------------
# Text cleaning
# ---------------------------------------------------------------------------
CID = re.compile(r"\(cid:\d+\)")          # PDF symbols pdfplumber can't decode, e.g. bullets -> "(cid:127)"
BULLET_START = re.compile(r"^\s*(\(cid:\d+\)|[•▪◦●\-–*])\s*")
NUMBERED_HEADING = re.compile(r"^\d+(\.\d+)*\.?\s+[A-Z]")   # e.g. "2. Scope of Work"


def clean(text, keep_lines=False):
    """Remove undecodable symbols, non-breaking spaces and repeated spaces.
    keep_lines=True keeps line breaks (used for table rows)."""
    text = CID.sub("", str(text)).replace("\u00a0", " ")
    if keep_lines:
        lines = (re.sub(r"[ \t]+", " ", ln).strip() for ln in text.splitlines())
        return "\n".join(ln for ln in lines if ln)
    return re.sub(r"\s+", " ", text).strip()


def _make_block(blocks, kind, text, section):
    text = clean(text, keep_lines=(kind == "table"))
    if text:
        blocks.append({"id": f"B{len(blocks) + 1:03d}", "type": kind, "section": section, "text": text})


def _finish(blocks, path, fmt):
    """Package blocks into one result, plus a readable full text for the LLM."""
    lines = []
    for b in blocks:
        prefix = {"heading": "## ", "bullet": "• "}.get(b["type"], "")
        lines.append(prefix + b["text"])
    full_text = "\n\n".join(lines)
    return {
        "source": Path(path).name,
        "format": fmt,
        "sections": [b["text"] for b in blocks if b["type"] == "heading"],
        "blocks": blocks,
        "full_text": full_text,
        "word_count": len(full_text.split()),
    }


# ---------------------------------------------------------------------------
# RFP: Word (.docx)
# ---------------------------------------------------------------------------
def _docx_items(doc):
    """Paragraphs AND tables, in the order they appear in the document."""
    for child in doc.element.body.iterchildren():
        tag = child.tag.split("}")[-1]
        if tag == "p":
            yield Paragraph(child, doc)
        elif tag == "tbl":
            yield Table(child, doc)


def parse_docx(path):
    doc = Document(check_allowed(path))
    blocks, section = [], "Front matter"
    for item in _docx_items(doc):
        if isinstance(item, Table):
            rows = [" | ".join(clean(c.text) for c in row.cells) for row in item.rows]
            _make_block(blocks, "table", "\n".join(rows), section)
            continue
        style = item.style.name if item.style is not None else ""
        has_numbering = item._p.pPr is not None and item._p.pPr.numPr is not None
        if style.startswith("Heading"):
            section = clean(item.text) or section
            _make_block(blocks, "heading", item.text, section)
        elif style == "Title":
            _make_block(blocks, "paragraph", item.text, section)
        elif "List" in style or has_numbering:
            _make_block(blocks, "bullet", item.text, section)
        else:
            _make_block(blocks, "paragraph", item.text, section)
    return _finish(blocks, path, "docx")


# ---------------------------------------------------------------------------
# RFP: PDF
# ---------------------------------------------------------------------------
def parse_pdf(path):
    """
    PDFs store text as LINES, not paragraphs, so we rebuild them:
      - a numbered short line ("2. Scope of Work") starts a new section
      - a line starting with a bullet symbol starts a new bullet
      - a short line ending in . : ! ? closes the current paragraph
      - otherwise the line continues the current paragraph
    """
    with pdfplumber.open(check_allowed(path)) as pdf:
        lines = [ln for page in pdf.pages for ln in (page.extract_text() or "").splitlines() if ln.strip()]
    if not lines:
        raise ValueError(f"No text found in '{path}' (is it a scanned PDF?)")

    full_width = max(len(ln) for ln in lines)
    blocks, section = [], "Front matter"
    current, kind = [], "paragraph"

    def flush():
        nonlocal current
        if current:
            _make_block(blocks, kind, " ".join(current), section)
            current = []

    for ln in lines:
        text = ln.strip()
        if NUMBERED_HEADING.match(text) and len(text) < 80 and not text.endswith("."):
            flush()
            section = clean(text)
            _make_block(blocks, "heading", text, section)
            kind = "paragraph"
            continue
        if BULLET_START.match(text):
            flush()
            kind = "bullet"
            text = BULLET_START.sub("", text)
        current.append(text)
        if text.endswith((".", ":", "!", "?")) and len(ln) < 0.85 * full_width:
            flush()
            kind = "paragraph"
    flush()
    return _finish(blocks, path, "pdf")


def parse_rfp(path):
    """Read an RFP from .pdf or .docx (chosen by file extension)."""
    check_allowed(path)          # safety guard ALWAYS runs first, whatever the file type
    suffix = Path(path).suffix.lower()
    if suffix == ".pdf":
        return parse_pdf(path)
    if suffix == ".docx":
        return parse_docx(path)
    raise ValueError(f"Unsupported RFP format '{suffix}'. Use .pdf or .docx (save old .doc files as .docx).")


# ---------------------------------------------------------------------------
# Proposal deck: PowerPoint (.pptx)
# ---------------------------------------------------------------------------
def _walk_shapes(shapes):
    """All shapes in reading order (top to bottom, left to right), opening groups."""
    for shape in sorted(shapes, key=lambda s: ((s.top or 0), (s.left or 0))):
        if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
            yield from _walk_shapes(shape.shapes)
        else:
            yield shape


def parse_deck(path):
    """
    One entry per slide: title, body lines, tables and speaker notes.
    'visible_text' = what the audience sees (title + body + tables).
    Speaker notes are kept separately: the client never sees them, so later
    stages can decide whether they count as "addressing" a requirement.
    """
    check_allowed(path)          # safety guard ALWAYS runs first
    if Path(path).suffix.lower() != ".pptx":
        raise ValueError(f"Unsupported deck format '{Path(path).suffix}'. Use .pptx (save old .ppt files as .pptx).")
    prs = Presentation(path)
    slides = []
    for number, slide in enumerate(prs.slides, start=1):
        title_shape = slide.shapes.title
        title = clean(title_shape.text_frame.text) if title_shape is not None and title_shape.has_text_frame else ""
        body, tables = [], []
        for shape in _walk_shapes(slide.shapes):
            if title_shape is not None and shape.shape_id == title_shape.shape_id:
                continue
            if shape.has_table:
                tables.append([[clean(c.text) for c in row.cells] for row in shape.table.rows])
            elif shape.has_text_frame:
                body.extend(t for t in (clean(p.text) for p in shape.text_frame.paragraphs) if t)
        notes = ""
        if slide.has_notes_slide:
            notes = clean(slide.notes_slide.notes_text_frame.text)

        parts = [f"Title: {title}"] + [f"- {b}" for b in body]
        for table in tables:
            parts += ["Table:"] + [" | ".join(row) for row in table]
        slides.append({
            "slide_number": number,
            "title": title,
            "body": body,
            "tables": tables,
            "notes": notes,
            "visible_text": "\n".join(parts),
        })
    return {"source": Path(path).name, "slide_count": len(slides), "slides": slides}


In [ ]:
%%writefile src/extract.py
"""
extract.py - STAGE 2 of the tool: find every requirement in the RFP.

Principle: "AI proposes, Python verifies."
  1. Gemini reads the RFP (as numbered blocks from parsers.py) and returns each
     requirement with a category, a short restatement, the EXACT quote and the
     block it came from.
  2. Python checks Gemini's work:
       - the quote must really exist in the RFP (catches invented text)
       - the RFP section is taken from the parser, not from Gemini
       - near-duplicate requirements are removed
       - a requirement whose quote is NOT in the RFP is set aside as "unverified"
         (saved and shown for human review, but not passed on), so the tool never
         reports a gap for something the client did not actually ask for
"""
import csv
import difflib
import json
import re
from pathlib import Path

from src.llm import MODEL, ask_json

CATEGORIES = ["Scope", "Deliverable", "Timeline", "Proposal requirement"]

SYSTEM = (
    "You are an experienced bid manager at a management consultancy. "
    "You read client RFPs and list precisely what the client requires, without adding anything."
)

INSTRUCTIONS = """Below is an RFP, split into numbered blocks. Each block shows its ID, type and RFP section.

TASK: list every requirement the client places on the consultant or on the proposal.

CATEGORIES (use exactly one):
- "Scope": work or activities the consultant must carry out
- "Deliverable": tangible outputs the consultant must hand over (reports, portfolios, presentations, language/format of outputs)
- "Timeline": durations, deadlines and milestones of the engagement itself
- "Proposal requirement": what the proposal document must contain or how it must be structured (e.g. CVs, pricing format)

RULES:
1. Keep a single ask together, even if it has several aspects (e.g. "assess the data infrastructure and its governance" is ONE requirement; later stages check each aspect).
2. Split lists of separate items into separate requirements (e.g. "a report, alongside a portfolio" = TWO deliverables).
3. List each requirement ONCE. If an ask is repeated elsewhere (e.g. in evaluation criteria or a summary), cite the place where it is stated as an obligation.
4. Do NOT extract: background or context, evaluation criteria and their weights, submission logistics (how, where or when to submit; clarification questions), confidentiality or legal terms.
5. "source_quote": copy the exact words from ONE block, verbatim, at most about 40 words. Never paraphrase the quote.
6. "requirement": a short, clear restatement (at most 25 words) starting with a verb.

Return JSON only:
{"requirements": [{"category": "...", "requirement": "...", "source_quote": "...", "block_id": "B000"}],
 "skipped_sections": [{"section": "...", "reason": "..."}]}

RFP BLOCKS:
"""


# ---------------------------------------------------------------------------
# Prompt
# ---------------------------------------------------------------------------
def build_prompt(rfp):
    """Show Gemini every block with its ID, type and section, so it can cite them."""
    lines = [f"[{b['id']} | {b['type']} | {b['section']}] {b['text']}" for b in rfp["blocks"]]
    return INSTRUCTIONS + "\n".join(lines)


# ---------------------------------------------------------------------------
# Python checks on Gemini's answer
# ---------------------------------------------------------------------------
def _norm(text):
    """Lower case, punctuation removed, single spaces: for fair text comparison."""
    text = re.sub(r"[^\w\s%]", " ", str(text).lower())
    return re.sub(r"\s+", " ", text).strip()


def quote_found(quote, text, threshold=0.9):
    """True if (almost) all of the quote appears, in order, inside the text."""
    q, t = _norm(quote), _norm(text)
    if not q:
        return False
    if q in t:
        return True
    matcher = difflib.SequenceMatcher(None, q, t, autojunk=False)
    matched = sum(block.size for block in matcher.get_matching_blocks())
    return matched / len(q) >= threshold


def _category(value):
    """Map Gemini's category onto our four fixed categories."""
    v = str(value).lower()
    for name, key in [("Deliverable", "deliver"), ("Timeline", "time"),
                      ("Proposal requirement", "proposal"), ("Scope", "scope")]:
        if key in v:
            return name
    return "Other"


def _is_duplicate(a, b):
    """Two requirements are duplicates if their statements or quotes are nearly the same."""
    same_text = difflib.SequenceMatcher(None, _norm(a["requirement"]), _norm(b["requirement"])).ratio() >= 0.85
    qa, qb = _norm(a["source_quote"]), _norm(b["source_quote"])
    same_quote = bool(qa and qb) and (qa in qb or qb in qa)
    return same_text or same_quote


def verify(raw_items, rfp):
    """Check each requirement against the real RFP text; return (kept, removed_duplicates, unverified)."""
    blocks = {b["id"]: b for b in rfp["blocks"]}
    order = {b["id"]: i for i, b in enumerate(rfp["blocks"])}
    checked = []
    for item in raw_items:
        if not isinstance(item, dict):
            continue
        quote = str(item.get("source_quote") or "").strip()
        match = re.search(r"B\d+", str(item.get("block_id") or ""))
        block_id = match.group(0) if match else ""

        if block_id in blocks and quote_found(quote, blocks[block_id]["text"]):
            status = "verified"
        else:
            found = next((b["id"] for b in rfp["blocks"] if quote_found(quote, b["text"])), None)
            if found:
                block_id, status = found, "verified (block corrected)"
            else:
                status = "NOT FOUND in RFP - review"

        checked.append({
            "category": _category(item.get("category")),
            "requirement": str(item.get("requirement") or "").strip(),
            "source_quote": quote,
            "block_id": block_id if block_id in blocks else "",
            "section": blocks[block_id]["section"] if block_id in blocks else "",
            "quote_check": status,
        })

    # set aside requirements whose quote is not in the RFP
    unverified = [r for r in checked if r["quote_check"].startswith("NOT FOUND")]
    checked = [r for r in checked if not r["quote_check"].startswith("NOT FOUND")]

    # keep RFP order, then remove near-duplicates (first one wins)
    checked.sort(key=lambda r: order.get(r["block_id"], 10**6))
    kept, removed = [], []
    for req in checked:
        twin = next((k for k in kept if _is_duplicate(k, req)), None)
        if twin:
            removed.append({**req, "duplicate_of": twin["requirement"]})
        else:
            kept.append(req)
    for i, req in enumerate(kept, start=1):
        req["id"] = f"R{i:02d}"
    return kept, removed, unverified


# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------
def extract_requirements(rfp):
    """Parsed RFP in, verified requirement list out."""
    reply = ask_json(build_prompt(rfp), system=SYSTEM, temperature=0.0, cache_tag="extract-v1")
    if isinstance(reply, list):
        reply = {"requirements": reply}
    raw = reply.get("requirements") if isinstance(reply, dict) else None
    if not isinstance(raw, list) or not raw:
        raise ValueError("Gemini returned no requirements.")
    kept, removed, unverified = verify(raw, rfp)
    if not kept:
        raise ValueError("None of Gemini's requirements could be verified against the RFP text.")
    return {
        "rfp_source": rfp["source"],
        "model": MODEL,
        "requirements": kept,
        "removed_duplicates": removed,
        "unverified": unverified,
        "skipped_sections": reply.get("skipped_sections") or [],
    }


def save_requirements(result, folder="output"):
    """Save the full result as JSON, and the requirement list as CSV."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "requirements.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    fields = ["id", "category", "requirement", "section", "block_id", "source_quote", "quote_check"]
    with open(folder / "requirements.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(result["requirements"])
    return folder / "requirements.json", folder / "requirements.csv"


In [ ]:
%%writefile src/retrieve.py
"""
retrieve.py - STAGE 3 of the tool: rank the slides by how relevant they are to
each requirement, using a local embedding model (free, runs on CPU, no API).

An embedding model turns a piece of text into a list of numbers (a "vector")
that captures its MEANING. Texts with similar meaning get similar vectors, even
when they use different words ("pilots" vs "rapid prototypes"). We compare the
requirement's vector with each slide's vector (cosine similarity, 0 to 1).

How the ranking is used (decision based on a measurement on Day 4):
  - deck of up to MAX_SLIDES_FOR_JUDGE slides -> the judge sees ALL slides,
    most relevant first (retrieval orders them, it never hides any)
  - bigger deck -> the judge sees only the MAX_SLIDES_FOR_JUDGE most relevant
On our test deck, a top-5 shortlist missed the right slide for 1 of 14
requirements, so a hard shortlist is only used when the deck is large.
"""
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
MAX_SLIDES_FOR_JUDGE = 25


class SlideRetriever:
    def __init__(self, deck, model_name=EMBEDDING_MODEL):
        self.slides = deck["slides"]
        self.model = None
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(model_name, device="cpu")
            texts = [s["visible_text"] for s in self.slides]   # what the client sees; notes excluded
            self.vectors = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
            self.method = f"embeddings ({model_name.split('/')[-1]})"
        except Exception as e:
            # Safe fallback: without embeddings the judge still sees every slide, in deck order
            self.method = f"none: embedding model unavailable ({str(e)[:100]}); slides kept in deck order"

    def rank(self, requirement, limit=MAX_SLIDES_FOR_JUDGE):
        """Return [{'slide_number': n, 'score': s}, ...], most relevant first, at most `limit` slides."""
        if self.model is None:
            ranked = [{"slide_number": s["slide_number"], "score": None} for s in self.slides]
        else:
            query = f"{requirement['requirement']} {requirement['source_quote']}"
            q = self.model.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
            scores = self.vectors @ q          # cosine similarity (vectors are normalised)
            order = np.argsort(-scores)
            ranked = [{"slide_number": self.slides[i]["slide_number"], "score": round(float(scores[i]), 3)}
                      for i in order]
        return ranked[:limit]


In [ ]:
%%writefile src/judge.py
"""
judge.py - STAGE 4 of the tool: decide, for each requirement, whether the deck
addresses it: Covered / Partially covered / Not covered.

Principle (same as Day 3): "AI proposes, Python verifies."
  1. Gemini reads ONE requirement plus the slides (most relevant first, visible
     text only) and returns a verdict, the slide numbers, an exact evidence
     quote, what is missing, a short reason and a confidence level.
  2. Python checks the verdict:
       - the status must be one of the three allowed values
       - the slide numbers must exist and must be among the slides shown
       - the evidence quote must really appear on the cited slides
     Anything that fails a check is flagged "needs review" (never silently fixed).
"""
import csv
import json
import re
from pathlib import Path

from src import llm
from src.extract import quote_found

STATUSES = ["Covered", "Partially covered", "Not covered"]

SYSTEM = (
    "You are a meticulous bid reviewer at a management consultancy. Before a proposal is "
    "submitted, you check whether the deck genuinely addresses each client requirement. "
    "You only give credit for what the slides actually commit to."
)

INSTRUCTIONS = """Decide whether the proposal deck addresses the client requirement below.

STATUS (choose exactly one):
- "Covered": the slides commit to ALL parts of the requirement. Different wording is fine.
- "Partially covered": the slides commit to SOME parts, but at least one part is missing.
- "Not covered": no slide commits to this requirement.

RULES:
1. Judge meaning, not shared words. A slide that merely mentions similar words without committing to this
   requirement for this client does not count (e.g. general claims about the firm, work done for other
   clients, or commercial terms).
2. Check EVERY part of the requirement (e.g. a number or quantity, a second element joined by "and",
   a format or language, a cost element, a deadline). If any part is missing, it is at most "Partially covered".
3. Do not infer anything the slides do not explicitly state.
4. "slides": numbers of the slides that address the requirement (fully or partly); [] if Not covered.
5. "evidence": copy the exact words from the most relevant slide (at most about 30 words); "" if Not covered.
6. "missing": what is not addressed; "" if Covered.

Return JSON only:
{"status": "...", "slides": [0], "evidence": "...", "missing": "...", "reasoning": "one or two sentences", "confidence": "High|Medium|Low"}
"""


def build_prompt(requirement, slides_by_number, ranked):
    parts = [INSTRUCTIONS,
             f"REQUIREMENT ({requirement['category']}): {requirement['requirement']}",
             f'Exact RFP wording: "{requirement["source_quote"]}"',
             "",
             "PROPOSAL SLIDES (visible text only, most relevant first):"]
    for r in ranked:
        slide = slides_by_number[r["slide_number"]]
        parts.append(f"=== Slide {slide['slide_number']} ===\n{slide['visible_text']}")
    return "\n".join(parts)


def _status(value):
    v = str(value).strip().lower()
    if v.startswith("partial"):
        return "Partially covered"
    if v.startswith("not") or v in ("missing", "uncovered", "no"):
        return "Not covered"
    if v.startswith("covered") or v in ("fully covered", "yes"):
        return "Covered"
    return None


def _slide_numbers(value):
    items = value if isinstance(value, list) else [value]
    numbers = []
    for item in items:
        for n in re.findall(r"\d+", str(item)):
            if int(n) not in numbers:
                numbers.append(int(n))
    return numbers


def judge_requirement(requirement, deck, retriever):
    """Judge one requirement. Returns a verified result record."""
    slides_by_number = {s["slide_number"]: s for s in deck["slides"]}
    ranked = retriever.rank(requirement)
    shown = [r["slide_number"] for r in ranked]

    reply = llm.ask_json(build_prompt(requirement, slides_by_number, ranked),
                         system=SYSTEM, temperature=0.0, cache_tag="judge-v1")
    reply = reply if isinstance(reply, dict) else {}

    flags = []
    status = _status(reply.get("status"))
    if status is None:
        flags.append(f"unclear status '{reply.get('status')}'")
        status = "Needs review"

    cited = _slide_numbers(reply.get("slides", []))
    unknown = [n for n in cited if n not in shown]
    if unknown:
        flags.append(f"cited slides not shown to the judge: {unknown}")
    cited = [n for n in cited if n in shown]

    evidence = str(reply.get("evidence") or "").strip()
    if status == "Not covered":
        cited, evidence_check = [], "n/a"
    elif not cited:
        evidence_check = "no slides cited"
        flags.append("verdict gives credit but cites no valid slide")
    elif evidence and any(quote_found(evidence, slides_by_number[n]["visible_text"], threshold=0.85) for n in cited):
        evidence_check = "verified"
    else:
        evidence_check = "NOT FOUND on cited slides"
        flags.append("evidence quote not found on the cited slides")

    return {
        "id": requirement["id"],
        "category": requirement["category"],
        "requirement": requirement["requirement"],
        "rfp_section": requirement["section"],
        "rfp_quote": requirement["source_quote"],
        "status": status,
        "slides": cited,
        "evidence": evidence,
        "evidence_check": evidence_check,
        "missing": str(reply.get("missing") or "").strip(),
        "reasoning": str(reply.get("reasoning") or "").strip(),
        "confidence": str(reply.get("confidence") or "").strip().capitalize() or "Unknown",
        "needs_review": bool(flags),
        "review_reasons": "; ".join(flags),
        "top_retrieved": [r["slide_number"] for r in ranked[:3]],
        "model": llm.last_model_used,
    }


def judge_all(requirements, deck, retriever, progress=print):
    """Judge every requirement, one Gemini call each (cached, so reruns are free)."""
    results = []
    for i, req in enumerate(requirements, start=1):
        result = judge_requirement(req, deck, retriever)
        results.append(result)
        slides = ", ".join(map(str, result["slides"])) or "-"
        flag = "  ⚠️ review" if result["needs_review"] else ""
        progress(f"[{i}/{len(requirements)}] {result['id']} {result['status']:<18} slides: {slides:<8}{flag}")
    return results


def save_judgements(results, folder="output"):
    """Save all verdicts as JSON (full detail) and CSV (one row per requirement)."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "judgements.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
    fields = ["id", "category", "requirement", "status", "slides", "evidence", "evidence_check", "missing",
              "reasoning", "confidence", "needs_review", "review_reasons", "rfp_section", "rfp_quote", "model"]
    with open(folder / "judgements.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        for r in results:
            writer.writerow({**r, "slides": ", ".join(map(str, r["slides"]))})
    return folder / "judgements.json", folder / "judgements.csv"


In [ ]:
%%writefile evaluation/__init__.py
# Evaluation code: the ONLY part of the project allowed to read test_data/ground_truth/


In [ ]:
%%writefile evaluation/check_extraction.py
"""
check_extraction.py - EVALUATION of stage 2 (requirement extraction).

This is NOT part of the tool. It is the "exam marking" step and the only kind
of code allowed to read the answer key in test_data/ground_truth/.

Question it answers: of the 18 requirements we planted in the RFP,
how many did the tool find (recall), and what else did it list?

Matching method (simple and transparent): for each planted requirement, find
the extracted requirement that shares the most content words with it.
score = shared content words / content words in the planted requirement.
A score of 0.5 or more counts as "found". Every match is printed, so a human
can check the matching itself.
"""
import json
import re
from pathlib import Path

STOPWORDS = set("""a an the and or of to for in on at by with from as is are be been all each
its it this that these those any into than then their our your shall must will should can
including include includes both least within over per""".split())


def content_words(text):
    """Meaningful words only (numbers are kept, e.g. '16', '500')."""
    return {w for w in re.findall(r"[a-z0-9]+", str(text).lower())
            if w.isdigit() or (len(w) > 2 and w not in STOPWORDS)}


def overlap(planted_text, extracted):
    planted = content_words(planted_text)
    found = content_words(extracted["source_quote"] + " " + extracted["requirement"])
    return len(planted & found) / len(planted) if planted else 0.0


def evaluate_extraction(result, answer_key_path="test_data/ground_truth/answer_key.json", threshold=0.5):
    planted = json.loads(Path(answer_key_path).read_text(encoding="utf-8"))
    extracted = result["requirements"]
    rows, used = [], {}
    for gt in planted:
        scored = sorted(((overlap(gt["text"], r), r) for r in extracted), key=lambda x: -x[0])
        score, best = scored[0] if scored else (0.0, None)
        hit = best is not None and score >= threshold
        if hit:
            used.setdefault(best["id"], []).append(gt["id"])
        rows.append({"planted_id": gt["id"], "category": gt["category"], "planted": gt["text"],
                     "found": hit, "match_id": best["id"] if hit else "", "score": round(score, 2),
                     "category_ok": hit and best["category"] == gt["category"]})
    merged = {rid: ids for rid, ids in used.items() if len(ids) > 1}
    extras = [r for r in extracted if r["id"] not in used]
    found = sum(r["found"] for r in rows)
    return {"rows": rows, "found": found, "total": len(planted), "recall": found / len(planted),
            "category_agreement": sum(r["category_ok"] for r in rows) / max(found, 1),
            "merged": merged, "extras": extras}


In [ ]:
%%writefile evaluation/check_coverage.py
"""
check_coverage.py - EVALUATION of stages 3-4 (retrieval and judge).

NOT part of the tool: the "exam marking" step, allowed to read ground_truth/.
Scored ONLY on the 18 planted requirements (decision 39: the answer key was
fixed before the documents existed). Extra requirements are reported, not scored.

Measures:
  - status accuracy: judge's verdict == planted status (Covered / Partial / Not covered)
  - gap detection: of the 8 planted gaps (4 partial + 4 missing), how many the
    tool flags as NOT fully covered; and false alarms among the 10 covered ones
  - slide accuracy: for covered/partial, did the judge cite a correct slide?
                    for not covered, did it cite no slide?
  - retrieval rank: position of the correct slide in the embedding ranking
"""
import json
from pathlib import Path

from evaluation.check_extraction import evaluate_extraction

STATUSES = ["Covered", "Partially covered", "Not covered"]


def evaluate_coverage(judgements, extraction_result, answer_key_path="test_data/ground_truth/answer_key.json"):
    planted = {g["id"]: g for g in json.loads(Path(answer_key_path).read_text(encoding="utf-8"))}
    links = evaluate_extraction(extraction_result, answer_key_path)   # planted ID -> extracted R-ID
    by_id = {j["id"]: j for j in judgements}

    rows = []
    for link in links["rows"]:
        gt = planted[link["planted_id"]]
        j = by_id.get(link["match_id"])
        predicted = j["status"] if j else "Not extracted"
        pred_slides = j["slides"] if j else []
        if gt["expected_status"] == "Not covered":
            slide_ok = not pred_slides
        else:
            slide_ok = bool(set(gt["expected_slides"]) & set(pred_slides))
        rows.append({
            "planted_id": gt["id"], "req_id": link["match_id"], "requirement": gt["text"],
            "expected": gt["expected_status"], "predicted": predicted,
            "status_ok": predicted == gt["expected_status"],
            "expected_slides": gt["expected_slides"], "predicted_slides": pred_slides,
            "slide_ok": slide_ok,
        })

    gaps = [r for r in rows if r["expected"] != "Covered"]
    covered = [r for r in rows if r["expected"] == "Covered"]
    confusion = {e: {p: sum(r["expected"] == e and r["predicted"] == p for r in rows) for p in STATUSES}
                 for e in STATUSES}
    scored_ids = {r["req_id"] for r in rows}
    return {
        "rows": rows,
        "total": len(rows),
        "status_correct": sum(r["status_ok"] for r in rows),
        "gaps_total": len(gaps),
        "gaps_caught": sum(r["predicted"] != "Covered" for r in gaps),
        "covered_total": len(covered),
        "false_alarms": sum(r["predicted"] != "Covered" for r in covered),
        "slide_correct": sum(r["slide_ok"] for r in rows),
        "confusion": confusion,
        "extras": [j for j in judgements if j["id"] not in scored_ids],
    }


def retrieval_ranks(retriever, requirements, extraction_result, answer_key_path="test_data/ground_truth/answer_key.json"):
    """For each planted requirement that has a correct slide: where does it rank? (1 = top)"""
    planted = {g["id"]: g for g in json.loads(Path(answer_key_path).read_text(encoding="utf-8"))}
    links = evaluate_extraction(extraction_result, answer_key_path)
    req_by_id = {r["id"]: r for r in requirements}
    ranks = {}
    for link in links["rows"]:
        gt = planted[link["planted_id"]]
        if not gt["expected_slides"] or link["match_id"] not in req_by_id:
            continue
        order = [r["slide_number"] for r in retriever.rank(req_by_id[link["match_id"]], limit=10**6)]
        ranks[gt["id"]] = min(order.index(n) + 1 for n in gt["expected_slides"] if n in order)
    return ranks


In [ ]:
%%writefile cache/a72e50dcdb4ebe8a.json
{
  "model_used": "gemini-3.8-flash",
  "tag": "extract-v1",
  "prompt_preview": "Below is an RFP, split into numbered blocks. Each block shows its ID, type and RFP section.\n\nTASK: list every requirement the client places on the consultant or on the proposal.\n\nCATEGORIES (use exactly one):\n- \"Scope\": work or activities the consultant must carry out\n- \"Deliverable\": tangible outpu",
  "response": {
    "requirements": [
      {
        "category": "Timeline",
        "requirement": "Complete the full engagement within 16 weeks from commencement.",
        "source_quote": "The appointed consultant shall complete the full engagement within 16 weeks from the official commencement date.",
        "block_id": "B007"
      },
      {
        "category": "Scope",
        "requirement": "Benchmark the group's AI maturity against at least three regional and global logistics peers.",
        "source_quote": "the consultant shall benchmark the group's AI maturity against at least three regional and global logistics peers.",
        "block_id": "B008"
      },
      {
        "category": "Scope",
        "requirement": "Assess the current AI and analytics maturity of all five business units.",
        "source_quote": "the consultant shall thoroughly assess the current AI and analytics maturity of all five business units (Ports, Freight, Warehousing, Customs Brokerage, Corporate Services).",
        "block_id": "B008"
      },
      {
        "category": "Scope",
        "requirement": "Assess the readiness of data infrastructure and data governance including ownership, quality, and policies.",
        "source_quote": "the consultant shall assess the readiness of the group's data infrastructure and its data governance, encompassing data ownership, data quality standards, and data policies.",
        "block_id": "B009"
      },
      {
        "category": "Scope",
        "requirement": "Identify at least 20 AI use cases and prioritise them by value and feasibility with financial estimates.",
        "source_quote": "the consultant shall identify at least 20 AI use cases and prioritise them by business value and feasibility, with an estimated financial impact for each.",
        "block_id": "B009"
      },
      {
        "category": "Scope",
        "requirement": "Design an AI target operating model, including an AI Centre of Excellence, roles, and decision rights.",
        "source_quote": "The consultant shall design an AI target operating model, including an AI Centre of Excellence, key roles, and decision rights.",
        "block_id": "B011"
      },
      {
        "category": "Scope",
        "requirement": "Develop a Responsible AI framework aligned with the UAE's national AI ethics principles.",
        "source_quote": "The consultant shall develop a Responsible AI framework aligned with the UAE's national AI ethics principles.",
        "block_id": "B012"
      },
      {
        "category": "Scope",
        "requirement": "Design a change management and AI upskilling programme for approximately 500 employees.",
        "source_quote": "The consultant shall design a change management and AI upskilling programme reaching approximately 500 employees.",
        "block_id": "B013"
      },
      {
        "category": "Deliverable",
        "requirement": "Deliver two proof-of-concept pilots for top-priority use cases.",
        "source_quote": "The consultant shall deliver two proof-of-concept pilots for top-priority use cases.",
        "block_id": "B014"
      },
      {
        "category": "Timeline",
        "requirement": "Mobilise the team and hold the project kick-off within two weeks of contract signature.",
        "source_quote": "The consultant shall mobilise the team and hold the project kick-off within two weeks of contract signature.",
        "block_id": "B016"
      },
      {
        "category": "Timeline",
        "requirement": "Hold an interim Steering Committee review at week 8 to evaluate preliminary findings and adjustments.",
        "source_quote": "the consultant shall hold an interim Steering Committee review at week 8 to evaluate preliminary findings, review initial use-case prioritisation, and adjust project trajectories as necessary.",
        "block_id": "B017"
      },
      {
        "category": "Deliverable",
        "requirement": "Provide all final deliverables in both Arabic and English.",
        "source_quote": "All final deliverables provided in both Arabic and English.",
        "block_id": "B019"
      },
      {
        "category": "Deliverable",
        "requirement": "Deliver a current-state assessment report detailing maturity across all operational domains.",
        "source_quote": "The mandated outputs must include a current-state assessment report detailing maturity across all operational domains",
        "block_id": "B020"
      },
      {
        "category": "Deliverable",
        "requirement": "Provide a prioritised AI use-case portfolio.",
        "source_quote": "alongside a prioritised AI use-case portfolio.",
        "block_id": "B020"
      },
      {
        "category": "Deliverable",
        "requirement": "Formulate a three-year AI roadmap including investment estimates by year.",
        "source_quote": "the consultant shall formulate a three-year AI roadmap including investment estimates by year.",
        "block_id": "B020"
      },
      {
        "category": "Deliverable",
        "requirement": "Deliver a final presentation to the Executive Board summarising the strategy, governance, and pilot outcomes.",
        "source_quote": "a final presentation to the Executive Board, summarising the strategic roadmap, governance models, and pilot outcomes.",
        "block_id": "B021"
      },
      {
        "category": "Proposal requirement",
        "requirement": "Include detailed company profiles in the proposal submission.",
        "source_quote": "submissions must contain detailed company profiles",
        "block_id": "B023"
      },
      {
        "category": "Proposal requirement",
        "requirement": "Demonstrate logistics sector experience in the proposal.",
        "source_quote": "demonstrable logistics sector experience",
        "block_id": "B023"
      },
      {
        "category": "Proposal requirement",
        "requirement": "Include comprehensive methodological approaches within the proposal.",
        "source_quote": "and comprehensive methodological approaches.",
        "block_id": "B023"
      },
      {
        "category": "Proposal requirement",
        "requirement": "Provide CVs of the proposed team, including at least two Arabic-speaking consultants.",
        "source_quote": "the consultant shall provide CVs of the proposed team, including at least two Arabic-speaking consultants.",
        "block_id": "B024"
      },
      {
        "category": "Proposal requirement",
        "requirement": "Provide fixed-fee pricing broken down by project phase.",
        "source_quote": "Bidders must also provide fixed-fee pricing broken down by project phase.",
        "block_id": "B024"
      }
    ],
    "skipped_sections": [
      {
        "section": "1. Introduction and Background",
        "reason": "Contains background and organizational context."
      },
      {
        "section": "6. Evaluation Criteria",
        "reason": "Contains evaluation criteria and scoring weights."
      },
      {
        "section": "7. Instructions and Important Dates",
        "reason": "Contains submission logistics, submission deadlines, and clarification question procedures."
      },
      {
        "section": "8. Confidentiality",
        "reason": "Contains confidentiality and legal terms."
      }
    ]
  }
}

In [ ]:
# ===== STEP 2: check all code files were saved =====
files = ['requirements.txt', 'src/llm.py', 'src/parsers.py', 'src/extract.py', 'src/retrieve.py', 'src/judge.py',
         'evaluation/__init__.py', 'evaluation/check_extraction.py', 'evaluation/check_coverage.py',
         'cache/a72e50dcdb4ebe8a.json']
missing = [f for f in files if not os.path.isfile(f) or os.path.getsize(f) == 0]
if missing:
    fail(2, f'files not saved: {missing}. Run the cells above this one')
print(f'{len(files)} files saved (incl. the official Day 3 extraction in the cache)')
print('✅ STEP 2 DONE: code files saved')

## Step 3: Install libraries

In [ ]:
# ===== STEP 3: install libraries and check they import =====
import subprocess, importlib
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
                        capture_output=True, text=True)
if result.returncode != 0:
    fail(3, 'pip install failed: ' + result.stderr.strip()[-300:])
missing = []
for module in ['pdfplumber', 'docx', 'pptx', 'google.genai', 'pandas', 'sentence_transformers']:
    try:
        importlib.import_module(module)
    except Exception as e:
        missing.append(f'{module} ({str(e)[:80]})')
if missing:
    fail(3, f'libraries not importable: {missing}')
print('(pip warnings about dopamine-rl / gradio are harmless and hidden here)')
print('✅ STEP 3 DONE: libraries ready (incl. sentence-transformers)')

## Step 4: API key and models

In [ ]:
# ===== STEP 4: load the API key from Kaggle Secrets and choose the models =====
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('GEMINI_API_KEY')
except Exception as e:
    fail(4, f'could not read GEMINI_API_KEY ({str(e)[:120]}). Fix: Add-ons > Secrets > tick GEMINI_API_KEY, then Run All again')
if not key:
    fail(4, 'GEMINI_API_KEY is empty. Fix: Add-ons > Secrets > check the value')
os.environ['GEMINI_API_KEY'] = key
os.environ['GEMINI_MODEL'] = 'gemini-3.8-flash'                 # main model
os.environ['GEMINI_FALLBACK_MODELS'] = 'gemini-3.5-flash-lite'  # used automatically if the main one is busy
print(f'Key loaded from Kaggle Secrets ({len(key)} characters, not shown)')
print('Main model: gemini-3.8-flash | backup: gemini-3.5-flash-lite')
print('✅ STEP 4 DONE: key and models set')

## Step 5: Load the code

In [ ]:
# ===== STEP 5: load the code =====
import importlib, warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('google_genai').setLevel(logging.ERROR)
try:
    import src.llm, src.parsers, src.extract, src.retrieve, src.judge
    import evaluation.check_extraction, evaluation.check_coverage
    for m in (src.llm, src.parsers, src.extract, src.retrieve, src.judge,
              evaluation.check_extraction, evaluation.check_coverage):
        importlib.reload(m)                 # always use the versions just written
    from src.parsers import parse_rfp, parse_deck
    from src.extract import extract_requirements, save_requirements
    from src.retrieve import SlideRetriever, MAX_SLIDES_FOR_JUDGE
    from src.judge import judge_all, save_judgements
    from evaluation.check_coverage import evaluate_coverage, retrieval_ranks
except Exception as e:
    fail(5, f'code could not be loaded: {e}')
if src.llm.MODEL != 'gemini-3.8-flash':
    fail(5, f'unexpected model {src.llm.MODEL}; run Step 4 before Step 5')
print('Loaded: parsers, extract, retrieve, judge, evaluation | model:', src.llm.MODEL)
print('✅ STEP 5 DONE: code loaded')

## Step 6: Parse both files and get the requirements

In [ ]:
# ===== STEP 6: stages 1-2: parse both files and get the requirements (official Day 3 extraction) =====
import pandas as pd
pd.set_option('display.max_colwidth', 80)
try:
    rfp = parse_rfp('test_data/inputs/rfp_qamar.pdf')
    deck = parse_deck('test_data/inputs/proposal_qamar.pptx')
    extraction = extract_requirements(rfp)
except Exception as e:
    fail(6, f'parsing or extraction failed: {str(e)[:300]}')
save_requirements(extraction, 'output')
requirements = extraction['requirements']
print(f"RFP: {len(rfp['blocks'])} blocks | deck: {deck['slide_count']} slides | requirements: {len(requirements)}")
if str(src.llm.last_model_used).startswith('cache'):
    print('Official Day 3 extraction reused from the cache (no API call)')
else:
    print('⚠️ Extraction was NOT found in the cache, so Gemini extracted again (results may differ slightly from Day 3)')
print('✅ STEP 6 DONE: RFP and deck parsed, requirements ready')

## Step 7: Retrieval: rank slides by meaning

In [ ]:
# ===== STEP 7: stage 3: rank slides by meaning with a local embedding model =====
print('Loading the embedding model (first run downloads ~420 MB, about 20-60 seconds)...')
retriever = SlideRetriever(deck)
print('Retrieval method:', retriever.method)
if retriever.model is None:
    print('⚠️ Embeddings unavailable: the judge will still see every slide, in deck order (results unaffected for this deck).')
shown = min(deck['slide_count'], MAX_SLIDES_FOR_JUDGE)
print(f'The judge will see {shown} of {deck["slide_count"]} slides per requirement, most relevant first.')
print('\nTop 3 slides by meaning, for each requirement:')
for req in requirements:
    top = retriever.rank(req, limit=3)
    tops = ', '.join(f"{t['slide_number']} ({t['score']})" if t['score'] is not None else str(t['slide_number']) for t in top)
    print(f"  {req['id']} {req['requirement'][:58]:<58} -> slides {tops}")
print('✅ STEP 7 DONE: retrieval ready')

## Step 8: The LLM judge

In [ ]:
# ===== STEP 8: stage 4: the LLM judge (one Gemini call per requirement, then Python checks) =====
print(f'Judging {len(requirements)} requirements (about 6 seconds apart; cached answers are instant)...')
try:
    judgements = judge_all(requirements, deck, retriever)
except Exception as e:
    fail(8, f'judging failed: {str(e)[:300]}')
json_path, csv_path = save_judgements(judgements, 'output')
table = pd.DataFrame(judgements)[['id', 'category', 'requirement', 'status', 'slides', 'confidence', 'needs_review']]
display(table)
print('Verdicts:', table['status'].value_counts().to_dict())
print('Models used:', pd.Series([j['model'] for j in judgements]).value_counts().to_dict())
print(f'Saved: {json_path} and {csv_path}')
print(f'✅ STEP 8 DONE: {len(judgements)} requirements judged')

## Step 9: Quality checks (no answer key)

In [ ]:
# ===== STEP 9: quality checks (the tool's own checks; no answer key used) =====
warnings_found = []
def check(ok, label):
    print(('  ✅ ' if ok else '  ⚠️ ') + label)
    if not ok:
        warnings_found.append(label)

valid = {'Covered', 'Partially covered', 'Not covered'}
check(all(j['status'] in valid for j in judgements), 'every verdict is Covered, Partially covered or Not covered')
credited = [j for j in judgements if j['status'] in ('Covered', 'Partially covered')]
verified = [j for j in credited if j['evidence_check'] == 'verified']
check(len(verified) == len(credited), f'evidence quote found on the cited slides for {len(verified)}/{len(credited)} credited requirements')
flagged = [j for j in judgements if j['needs_review']]
check(not flagged, f'{len(flagged)} verdict(s) flagged for human review'
      + ''.join(f"\n       - {j['id']}: {j['review_reasons']}" for j in flagged))
models = {j['model'] for j in judgements}
print(f'  ℹ️ models used: {sorted(models)}')
gaps = [j for j in judgements if j['status'] != 'Covered']
print(f'  ℹ️ gaps reported by the tool: {len(gaps)} ('
      + ', '.join(f"{j['id']} {j['status'].split()[0]}" for j in gaps) + ')')
print(f'✅ STEP 9 DONE: quality checks complete ({len(warnings_found)} warning(s))')

## Step 10: Evaluation preview against the answer key

In [ ]:
# ===== STEP 10: EVALUATION preview against the answer key (reads ground_truth/; scored on the 18 planted only) =====
ev = evaluate_coverage(judgements, extraction)
rows = pd.DataFrame(ev['rows'])[['planted_id', 'req_id', 'expected', 'predicted', 'status_ok',
                                 'expected_slides', 'predicted_slides', 'slide_ok']]
display(rows)
print(f"Status accuracy:  {ev['status_correct']}/{ev['total']} planted requirements judged correctly")
print(f"Gaps caught:      {ev['gaps_caught']}/{ev['gaps_total']} planted gaps (partial or missing) flagged as not fully covered")
print(f"False alarms:     {ev['false_alarms']}/{ev['covered_total']} covered requirements wrongly flagged as gaps")
print(f"Slide accuracy:   {ev['slide_correct']}/{ev['total']} (correct slide cited; or no slide, when nothing covers it)")
print('\nConfusion table (rows = planted answer, columns = tool verdict):')
display(pd.DataFrame(ev['confusion']).T)
wrong = [r for r in ev['rows'] if not r['status_ok']]
if wrong:
    by_id = {j['id']: j for j in judgements}
    print('Disagreements, with the judge\'s reasoning:')
    for r in wrong:
        j = by_id.get(r['req_id'], {})
        print(f"  {r['planted_id']} expected {r['expected']}, tool said {r['predicted']} (slides {r['predicted_slides']}): {j.get('reasoning', '')}")
ranks = retrieval_ranks(retriever, requirements, extraction) if retriever.model is not None else {}
if ranks:
    print(f"\nRetrieval: correct slide ranked in the top 3 for {sum(v <= 3 for v in ranks.values())}/{len(ranks)}; "
          f"top 5 for {sum(v <= 5 for v in ranks.values())}/{len(ranks)} (ranks: {ranks})")
print('\nExtra requirements (reported, not scored):')
for e in ev['extras']:
    print(f"  {e['id']} {e['status']:<18} slides {e['slides']}: {e['requirement']}")
print(f"✅ STEP 10 DONE: status accuracy {ev['status_correct']}/{ev['total']}, gaps caught {ev['gaps_caught']}/{ev['gaps_total']}")

## Step 11: Download

In [ ]:
# ===== STEP 11: save everything as a zip to download =====
zip_path = shutil.make_archive('/kaggle/working/rfp_coverage_mapper_day4', 'zip', '/kaggle/working', 'rfp-coverage-mapper')
size_kb = os.path.getsize(zip_path) // 1024
if size_kb == 0:
    fail(11, 'zip file is empty')
print(f'Saved {zip_path} ({size_kb} KB): code + test data + cache + output/')
print('✅ STEP 11 DONE: download rfp_coverage_mapper_day4.zip from the Output panel')